# Part 6a — Defense Generation (target Llama-2-7B-chat only)

Sub-notebook 1 of 3 for Part 6. Splits the original `06_defenses.ipynb` so each vLLM model lives in its own kernel — fixes host-RAM OOMs caused by stacking target + Guard-1 + Guard-3 buffers in one process.

**Produces** (in `results_part6/intermediate/`):
- `phase1_baseline.json` — no-defense responses per attack
- `phase1_smoothllm_gen.json` — N=10 perturbed copies + responses per attack
- `phase1_ppl.json` — PPL filter pass/fail + threshold
- `phase1_eac_gen.json` — Erase-and-Check variant responses per attack

Next: run `06b_defenses_guard1.ipynb` (restart kernel first).

In [1]:
import os, sys
from pathlib import Path

# Self-locate jbb_common.py so kernel CWD doesn't matter (tml/ vs repo root).
_cwd = Path.cwd()
if (_cwd / "jbb_common.py").exists():
    pass
elif (_cwd / "tml" / "jbb_common.py").exists():
    os.chdir(_cwd / "tml")
else:
    raise FileNotFoundError(f"jbb_common.py not found from {_cwd}; run from tml/ or repo root.")
sys.path.insert(0, str(Path.cwd()))

from jbb_common import (
    setup_vllm_env, patch_jbb, hf_login, load_jbb_dataset,
)
setup_vllm_env()
patch_jbb()
hf_login()
goals, benign_goals, judge_df = load_jbb_dataset()
print(f"Loaded {len(goals)} harmful goals")

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generating harmful split: 0 examples [00:00, ? examples/s]

Generating harmful split: 100 examples [00:00, 3961.71 examples/s]

Generating benign split: 0 examples [00:00, ? examples/s]

Generating benign split: 100 examples [00:00, 20967.33 examples/s]

...

Generating benign split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating benign split: 100%|██████████| 100/100 [00:00<00:00, 22828.63 examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating test split: 100%|██████████| 300/300 [00:00<00:00, 14746.18 examples/s]

Loaded 100 harmful goals


In [2]:
import os, json, time, gc
import numpy as np
import torch
from transformers import AutoTokenizer
from vllm import SamplingParams
import jailbreakbench as jbb
from jailbreakbench.llm.vllm import LLMvLLM
from jailbreakbench.defenses.defenselib import perturbations

N_COPIES = 10
PERT_PCT = 10
REFUSAL_RESPONSE = "I'm sorry, I cannot assist with that request."
ATTACK_METHODS   = ["PAIR", "GCG", "JBC", "prompt_with_random_search"]
TARGET_MODEL_NAME = "llama-2-7b-chat-hf"
ERASE_LENGTH = 20

INTER_DIR = "results_part6/intermediate"
os.makedirs(INTER_DIR, exist_ok=True)

artifacts = {}
for m in ATTACK_METHODS:
    art = jbb.read_artifact(method=m, model_name=TARGET_MODEL_NAME)
    artifacts[m] = art
    n_with_prompt = sum(1 for jb in art.jailbreaks if jb.prompt)
    print(f"{m:30s}  usable={n_with_prompt}/{len(art.jailbreaks)}")

all_inputs = {
    m: [{"behavior": jb.behavior, "goal": jb.goal, "prompt": jb.prompt}
        for jb in art.jailbreaks if jb.prompt]
    for m, art in artifacts.items()
}

PAIR                            usable=4/100
GCG                             usable=100/100
JBC                             usable=100/100
prompt_with_random_search       usable=100/100


In [3]:
target_llm = LLMvLLM(model_name=TARGET_MODEL_NAME)

def build_chat(prompt_text):
    return [
        {"role": "system", "content": target_llm.system_prompt},
        {"role": "user",   "content": prompt_text},
    ]

INFO 06-05 16:51:46 [utils.py:261] non-default args: {'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'meta-llama/Llama-2-7b-chat-hf'}


INFO 06-05 16:52:01 [model.py:541] Resolved architecture: LlamaForCausalLM


INFO 06-05 16:52:01 [model.py:1561] Using max model len 4096


2026-06-05 16:52:02,114	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 06-05 16:52:02 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.


...

Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:07<00:00,  4.00s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:07<00:00,  3.67s/it]
(EngineCore_DP0 pid=605) 


(EngineCore_DP0 pid=605) INFO 06-05 16:52:46 [default_loader.py:291] Loading weights took 7.36 seconds


(EngineCore_DP0 pid=605) INFO 06-05 16:52:46 [gpu_model_runner.py:4118] Model loading took 12.55 GiB memory and 27.868433 seconds


(EngineCore_DP0 pid=605) INFO 06-05 16:52:50 [gpu_worker.py:356] Available KV cache memory: 6.75 GiB
(EngineCore_DP0 pid=605) INFO 06-05 16:52:50 [kv_cache_utils.py:1307] GPU KV cache size: 13,808 tokens
(EngineCore_DP0 pid=605) INFO 06-05 16:52:50 [kv_cache_utils.py:1312] Maximum concurrency for 4,096 tokens per request: 3.37x
(EngineCore_DP0 pid=605) INFO 06-05 16:52:50 [core.py:272] init engine (profile, create kv cache, warmup model) took 3.64 seconds


(EngineCore_DP0 pid=605) INFO 06-05 16:52:51 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=605) WARNING 06-05 16:52:51 [vllm.py:669] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore_DP0 pid=605) INFO 06-05 16:52:51 [vllm.py:762] Cudagraph is disabled under eager mode
INFO 06-05 16:52:51 [llm.py:343] Supported tasks: ['generate']


In [4]:
# Baseline (no defense)
def run_no_defense(rows):
    chats = [build_chat(r["prompt"]) for r in rows]
    out = target_llm.query_llm(chats)
    return [{**rows[i], "response": out.responses[i]} for i in range(len(rows))]

baseline_results = {}
for m, rows in all_inputs.items():
    t0 = time.time()
    baseline_results[m] = run_no_defense(rows)
    print(f"baseline / {m:30s}  n={len(rows):3d}  {time.time()-t0:6.1f}s")

with open(f"{INTER_DIR}/phase1_baseline.json", "w") as f:
    json.dump(baseline_results, f)
print(f"Saved \u2192 {INTER_DIR}/phase1_baseline.json")

baseline / PAIR                            n=  4     5.9s


baseline / GCG                             n=100    13.5s


baseline / JBC                             n=100    12.1s


baseline / prompt_with_random_search       n=100    52.5s
Saved → results_part6/intermediate/phase1_baseline.json


In [5]:
# SmoothLLM generation (N=10 perturbed copies). Persist + free before next phase.
pert_fn = perturbations.RandomSwapPerturbation(q=PERT_PCT)

smoothllm_gen = {}
for m, rows in all_inputs.items():
    t0 = time.time()
    chats, owner, perturbed_texts = [], [], []
    for bi, r in enumerate(rows):
        for _ in range(N_COPIES):
            pt = pert_fn(r["prompt"])
            chats.append(build_chat(pt))
            owner.append(bi)
            perturbed_texts.append(pt)
    out = target_llm.query_llm(chats)
    gen = [{"row": r, "copies": []} for r in rows]
    for bi, pt, resp in zip(owner, perturbed_texts, out.responses):
        gen[bi]["copies"].append({"perturbed_prompt": pt, "response": resp})
    smoothllm_gen[m] = gen
    print(f"SmoothLLM gen / {m:30s}  n={len(rows):3d} x{N_COPIES}  {time.time()-t0:6.1f}s")

with open(f"{INTER_DIR}/phase1_smoothllm_gen.json", "w") as f:
    json.dump(smoothllm_gen, f)
print(f"Saved \u2192 {INTER_DIR}/phase1_smoothllm_gen.json")
del smoothllm_gen
gc.collect()

SmoothLLM gen / PAIR                            n=  4 x10    11.7s


SmoothLLM gen / GCG                             n=100 x10   137.0s


SmoothLLM gen / JBC                             n=100 x10   652.8s


SmoothLLM gen / prompt_with_random_search       n=100 x10   780.3s
Saved → results_part6/intermediate/phase1_smoothllm_gen.json


52

In [6]:
# Perplexity Filter via vLLM prompt_logprobs. Chunked to bound host RAM held by Logprob objects.
_ppl_sp = SamplingParams(temperature=0.0, max_tokens=1, prompt_logprobs=1)

def mean_token_ce(strings, chunk_size=16):
    vals = []
    for i in range(0, len(strings), chunk_size):
        chunk = strings[i:i + chunk_size]
        outs = target_llm.model.generate(chunk, _ppl_sp)
        for o in outs:
            lps = []
            for tid, lp_dict in zip(o.prompt_token_ids, o.prompt_logprobs):
                if lp_dict is None:
                    continue
                lp = lp_dict.get(tid)
                if lp is not None:
                    lps.append(lp.logprob)
            vals.append(-sum(lps) / max(len(lps), 1))
        del outs
        gc.collect()
    return vals

ppl_threshold = max(mean_token_ce(goals))
print(f"PPL threshold (max goal log-perplexity): {ppl_threshold:.3f}")

ppl_results_defense = {}
for m, rows in all_inputs.items():
    t0 = time.time()
    ces = mean_token_ce([r["prompt"] for r in rows])
    out_rows = []
    for r, ce, base in zip(rows, ces, baseline_results[m]):
        passed = ce <= ppl_threshold
        resp = base["response"] if passed else REFUSAL_RESPONSE
        out_rows.append({**r, "response": resp, "log_ppl": ce, "passed": passed})
    ppl_results_defense[m] = out_rows
    n_block = sum(1 for x in out_rows if not x["passed"])
    print(f"PerplexityFilter / {m:30s}  n={len(rows):3d}  blocked={n_block}  {time.time()-t0:6.1f}s")

with open(f"{INTER_DIR}/phase1_ppl.json", "w") as f:
    json.dump({"results": ppl_results_defense, "threshold": float(ppl_threshold)}, f)
print(f"Saved \u2192 {INTER_DIR}/phase1_ppl.json")
del ppl_results_defense, baseline_results
gc.collect()

Adding requests:   0%|          | 0/16 [00:00<?, ?it/s]

Adding requests: 100%|██████████| 16/16 [00:00<00:00, 3430.05it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   6%|▋         | 1/16 [00:04<01:01,  4.11s/it, est. speed input: 5.35 toks/s, output: 0.24 toks/s]

Processed prompts: 100%|██████████| 16/16 [00:04<00:00,  4.11s/it, est. speed input: 78.16 toks/s, output: 3.88 toks/s]

...

Processed prompts:  25%|██▌       | 1/4 [00:00<00:01,  1.57it/s, est. speed input: 881.54 toks/s, output: 1.57 toks/s]

Processed prompts: 100%|██████████| 4/4 [00:00<00:00,  1.57it/s, est. speed input: 3345.31 toks/s, output: 5.98 toks/s]

Processed prompts: 100%|██████████| 4/4 [00:00<00:00,  5.97it/s, est. speed input: 3345.31 toks/s, output: 5.98 toks/s]

PerplexityFilter / prompt_with_random_search       n=100  blocked=0    18.8s
Saved → results_part6/intermediate/phase1_ppl.json


32

In [7]:
# Erase-and-Check generation (suffix-erase up to 20 tokens).
eac_tok = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-chat-hf")

def erased_variants(prompt_text):
    ids = eac_tok.encode(prompt_text)
    variants = [prompt_text]
    for i in range(min(ERASE_LENGTH, len(ids))):
        variants.append(eac_tok.decode(ids[: -(i + 1)]))
    return variants

eac_gen = {}
for m, rows in all_inputs.items():
    t0 = time.time()
    chats, owner, variants_flat = [], [], []
    for bi, r in enumerate(rows):
        for v in erased_variants(r["prompt"]):
            chats.append(build_chat(v))
            owner.append(bi)
            variants_flat.append(v)
    out = target_llm.query_llm(chats)
    gen = [{"row": r, "candidates": []} for r in rows]
    for bi, v, resp in zip(owner, variants_flat, out.responses):
        gen[bi]["candidates"].append({"variant_prompt": v, "response": resp})
    eac_gen[m] = gen
    print(f"EraseAndCheck gen / {m:30s}  n={len(rows):3d}  candidates={len(chats):5d}  {time.time()-t0:6.1f}s")

with open(f"{INTER_DIR}/phase1_eac_gen.json", "w") as f:
    json.dump(eac_gen, f)
print(f"Saved \u2192 {INTER_DIR}/phase1_eac_gen.json")
del eac_gen, target_llm
gc.collect()
torch.cuda.empty_cache()
print(f"Free GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")
print("\nDone. Restart kernel, then run 06b_defenses_guard1.ipynb.")

EraseAndCheck gen / PAIR                            n=  4  candidates=   84     9.7s


EraseAndCheck gen / GCG                             n=100  candidates= 2100   206.0s


EraseAndCheck gen / JBC                             n=100  candidates= 2100   262.5s


EraseAndCheck gen / prompt_with_random_search       n=100  candidates= 2100   339.7s
Saved → results_part6/intermediate/phase1_eac_gen.json


[rank0]:[W605 17:35:03.398333213 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Free GPU memory: 25.1 GB

Done. Restart kernel, then run 06b_defenses_guard1.ipynb.
